In [54]:
import pandas as pd
import numpy as np


In [55]:
data_dir = "C:/Users/Admin/Desktop/Feature-Selection-Methods/data"
df = pd.read_csv(f"{data_dir}/Titanic-Dataset.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [56]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [57]:
df = df.drop(columns = "Cabin")
df.fillna({"Age" : df["Age"].mean()}, inplace = True)
df.fillna({"Embarked" : df["Embarked"].mode()[0]}, inplace = True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Embarked     891 non-null    str    
dtypes: float64(2), int64(5), str(4)
memory usage: 76.7 KB


In [58]:
# Features and target
X = df[['Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']]
y = df['Survived']

In [59]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical variables
X_encoded = X.copy()
for col in ['Sex','Embarked']:
    X_encoded[col] = LabelEncoder().fit_transform(X_encoded[col].astype(str))

## 1. Variance Threshold

In [60]:
from sklearn.feature_selection import VarianceThreshold

var_selector = VarianceThreshold(threshold=0.01)
X_var = var_selector.fit_transform(X_encoded)
var_features = X_encoded.columns[var_selector.get_support()]

## 2. Correlation Coefficient

In [61]:
X_corr = X_encoded.copy()
X_corr['Survived'] = y

correlations = X_corr.corr()['Survived'].drop('Survived')
print(correlations.sort_values(ascending=False))

Fare        0.257307
Parch       0.081629
SibSp      -0.035322
Age        -0.069809
Embarked   -0.167675
Pclass     -0.338481
Sex        -0.543351
Name: Survived, dtype: float64


## 3. Chi-Square Test

In [62]:
from sklearn.feature_selection import SelectKBest, chi2

chi2_selector = SelectKBest(chi2, k=3)
X_chi2 = chi2_selector.fit_transform(X_encoded, y)
chi2_features = X_encoded.columns[chi2_selector.get_support()]


## 4. ANOVA F-Test

In [63]:
from sklearn.feature_selection import f_classif

anova_selector = SelectKBest(f_classif, k=3)
X_anova = anova_selector.fit_transform(X_encoded, y)
anova_features = X_encoded.columns[anova_selector.get_support()]

## 5. Mutual Information

In [64]:
from sklearn.feature_selection import mutual_info_classif

mi_selector = SelectKBest(mutual_info_classif, k=3)
X_mi = mi_selector.fit_transform(X_encoded, y)
mi_features = X_encoded.columns[mi_selector.get_support()]

In [65]:
print("Variance Threshold:", var_features.tolist())
print("Chi-Square:", chi2_features.tolist())
print("ANOVA:", anova_features.tolist())
print("Mutual Info:", mi_features.tolist())

Variance Threshold: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
Chi-Square: ['Pclass', 'Sex', 'Fare']
ANOVA: ['Pclass', 'Sex', 'Fare']
Mutual Info: ['Pclass', 'Sex', 'Fare']


In [72]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X_encoded[mi_features], y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy with Mutual Info features:", accuracy_score(y_test, y_pred))

Accuracy with Mutual Info features: 0.7821229050279329


The test is not good though use al off these test

In [73]:
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy with Mutual Info features:", accuracy_score(y_test, y_pred))

Accuracy with Mutual Info features: 0.8100558659217877


The code without using test is better